In [7]:
# multi_taxi_routing.py
from collections import defaultdict
import heapq
import math
import itertools

# -----------------------
# Input (sample)
# -----------------------
N, M, P, W_min, S_kmph = 8, 9, 3, 30, 40
coords = {
    1: (0, 0),
    2: (30, 0),
    3: (80, 0),
    4: (30, 60),
    5: (120, 0),
    6: (30, 120),
    7: (60, 120),
    8: (110, 120)
}
# edges with distances (u,v) : distance_km
edge_list = {
    (1, 2): 30,
    (2, 3): 50,
    (2, 4): 60,
    (3, 5): 40,
    (4, 6): 70,
    (5, 7): 20,
    (6, 7): 30,
    (7, 8): 50,
    (3, 6): 90
}
# passenger trips (start -> dest)
trips = [(2, 7), (1, 8), (3, 4)]

# -----------------------
# Helpers
# -----------------------
time_per_km = 60.0 / S_kmph  # minutes per km

# build adjacency
adj = defaultdict(list)
for (u, v), d in edge_list.items():
    adj[u].append((v, d))
    adj[v].append((u, d))

def dijkstra_time(start):
    """Return (dist_map, parent_map) of shortest times (minutes) from start to all nodes."""
    inf = float('inf')
    dist = {n: inf for n in range(1, N+1)}
    parent = {}
    dist[start] = 0.0
    pq = [(0.0, start)]
    while pq:
        curd, u = heapq.heappop(pq)
        if curd > dist[u]:
            continue
        for v, d in adj[u]:
            t = d * time_per_km
            if curd + t < dist[v]:
                dist[v] = curd + t
                parent[v] = u
                heapq.heappush(pq, (dist[v], v))
    return dist, parent

def extract_path_from_parent(parent, src, dst):
    p = []
    cur = dst
    while cur != src:
        p.append(cur)
        cur = parent.get(cur, None)
        if cur is None:
            return None
    p.append(src)
    return p[::-1]

# For each trip, compute shortest path (Dijkstra from source or from goal)
# We'll compute Dijkstra from the goal to get shortest remaining times (admissible heuristic),
# and then extract the shortest path (we can run standard Dijkstra from source too).
def shortest_path(src, dst):
    # classic Dijkstra from src -> dst to extract path
    dist = {n: float('inf') for n in range(1, N+1)}
    parent = {}
    dist[src] = 0.0
    pq = [(0.0, src)]
    while pq:
        curd, u = heapq.heappop(pq)
        if u == dst:
            break
        if curd > dist[u]:
            continue
        for v, d in adj[u]:
            t = d * time_per_km
            if curd + t < dist[v]:
                dist[v] = curd + t
                parent[v] = u
                heapq.heappush(pq, (dist[v], v))
    return extract_path_from_parent(parent, src, dst), dist.get(dst, float('inf'))

# -----------------------
# Compute shortest paths for all trips
# -----------------------
paths = []
for s, t in trips:
    p, total = shortest_path(s, t)
    if p is None:
        raise RuntimeError(f"No path found {s}->{t}")
    paths.append(p)

# -----------------------
# Event-driven simulation for congestion
# -----------------------
# Events in PQ: (time, tie_break, type, taxi_id, info)
# type 'attempt' : info = edge_index (index in the path indicating edge path[i] -> path[i+1])
# type 'exit'    : info = normalized_edge_key (tuple)
event_q = []
counter = itertools.count()
# edge -> list of currently active taxi IDs (we maintain active list; removal occurs on exit events)
edge_active = defaultdict(list)
taxi_intervals = [[] for _ in range(len(paths))]  # per taxi list of (u,v,entry,exit)

# initialize: each taxi attempts its first edge at t=0 (if they have edges)
for tidx, path in enumerate(paths):
    if len(path) > 1:
        heapq.heappush(event_q, (0.0, next(counter), 'attempt', tidx, 0))
    else:
        # zero-length path (already at destination)
        taxi_intervals[tidx] = []

while event_q:
    time, _, etype, taxi, info = heapq.heappop(event_q)
    if etype == 'attempt':
        i_edge = info
        path = paths[taxi]
        if i_edge >= len(path) - 1:
            # reached destination; nothing to do
            continue
        u, v = path[i_edge], path[i_edge+1]
        key = tuple(sorted((u, v)))
        d = edge_list.get((u, v), edge_list.get((v, u)))
        edge_time = d * time_per_km
        # number of taxis currently on the edge at this moment
        active_count = len(edge_active[key])
        if active_count >= 2:
            # must wait W_min minutes then re-attempt
            heapq.heappush(event_q, (time + W_min, next(counter), 'attempt', taxi, i_edge))
        else:
            # enter edge now
            entry = time
            exit_time = entry + edge_time
            edge_active[key].append(taxi)
            taxi_intervals[taxi].append((u, v, entry, exit_time))
            # schedule exit event and next edge attempt at exit_time
            heapq.heappush(event_q, (exit_time, next(counter), 'exit', taxi, key))
            heapq.heappush(event_q, (exit_time, next(counter), 'attempt', taxi, i_edge + 1))

    else:  # 'exit' event
        key = info
        if taxi in edge_active[key]:
            edge_active[key].remove(taxi)

# -----------------------
# Print results
# -----------------------
total_completion = 0.0
makespan = 0.0
for i, path in enumerate(paths):
    print(f"Taxi {i+1}:")
    print(f"Passenger {path[0]}->{path[-1]}")
    print("Route: " + " -> ".join(str(x) for x in path))
    # find waits by checking gaps between arrival to node and next edge entry > 0
    total_time = 0.0
    for (u, v, entry, exit_t) in taxi_intervals[i]:
        print(f"  Edge {u}->{v}  Entry={entry:.1f}  Exit={exit_t:.1f}  time-on-edge={(exit_t-entry):.1f} min")
        total_time = exit_t
    if not taxi_intervals[i]:
        total_time = 0.0
    print(f"Total time = {total_time:.1f} minutes\n")
    total_completion += total_time
    makespan = max(makespan, total_time)

print(f"Total Completion Time = {total_completion:.1f} minutes")
print(f"Makespan (max taxi time) = {makespan:.1f} minutes")


Taxi 1:
Passenger 2->7
Route: 2 -> 3 -> 5 -> 7
  Edge 2->3  Entry=0.0  Exit=75.0  time-on-edge=75.0 min
  Edge 3->5  Entry=75.0  Exit=135.0  time-on-edge=60.0 min
  Edge 5->7  Entry=135.0  Exit=165.0  time-on-edge=30.0 min
Total time = 165.0 minutes

Taxi 2:
Passenger 1->8
Route: 1 -> 2 -> 3 -> 5 -> 7 -> 8
  Edge 1->2  Entry=0.0  Exit=45.0  time-on-edge=45.0 min
  Edge 2->3  Entry=75.0  Exit=150.0  time-on-edge=75.0 min
  Edge 3->5  Entry=150.0  Exit=210.0  time-on-edge=60.0 min
  Edge 5->7  Entry=210.0  Exit=240.0  time-on-edge=30.0 min
  Edge 7->8  Entry=240.0  Exit=315.0  time-on-edge=75.0 min
Total time = 315.0 minutes

Taxi 3:
Passenger 3->4
Route: 3 -> 2 -> 4
  Edge 3->2  Entry=0.0  Exit=75.0  time-on-edge=75.0 min
  Edge 2->4  Entry=75.0  Exit=165.0  time-on-edge=90.0 min
Total time = 165.0 minutes

Total Completion Time = 645.0 minutes
Makespan (max taxi time) = 315.0 minutes
